# Trabajo Integrador: Dise?o, Limpieza y An?lisis de Datos
## Semana 9, Corte 2
**Fecha entrega:** 2026-10-04 23:59
**Puntos:** 5.0

```sql
CREATE TABLE maquina (
    id INTEGER PRIMARY KEY,
    nombre VARCHAR(50) UNIQUE NOT NULL,
    linea VARCHAR(100),
    modelo VARCHAR(100),
    anio_instalacion INTEGER,
    capacidad_max INTEGER DEFAULT 1800
);

CREATE TABLE turno (
    id INTEGER PRIMARY KEY,
    fecha DATE NOT NULL,
    nombre_turno VARCHAR(20) CHECK (nombre_turno IN ('Ma?ana', 'Tarde', 'Noche')),
    UNIQUE(fecha, nombre_turno)
);

CREATE TABLE registro (
    id INTEGER PRIMARY KEY,
    maquina_id INTEGER NOT NULL,
    turno_id INTEGER NOT NULL,
    producidas INTEGER CHECK (producidas > 0),
    defectuosas INTEGER CHECK (defectuosas >= 0),
    temperatura_c DECIMAL(5,1),
    temp_imputada BOOLEAN DEFAULT FALSE,
    FOREIGN KEY (maquina_id) REFERENCES maquina(id),
    FOREIGN KEY (turno_id) REFERENCES turno(id)
);
```

Relaciones:
- M?quina 1?N Registro
- Turno 1?N Registro
- Cardinalidad: N:N (pivote: Registro)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sqlite3

df = pd.read_csv("../registro_produccion.csv")
print("Shape:", df.shape)
print("\nPrimeras filas:")
print(df.head(10))
print("\nTipos:")
print(df.info())
print("\nEstad?sticas:")
print(df.describe())

Matplotlib is building the font cache; this may take a moment.


Shape: (297, 6)

Primeras filas:
        fecha   turno maquina  producidas  defectuosas temperatura
0  07/09/2026  MAÑANA     M01         889         62.0       102.6
1  2026-09-07  Mañana    M-02        1574        146.0        23.9
2  2026-09-08  Mañana    M-03        1327        131.0        37.7
3  2026-09-09  Mañana    M-04        1776         74.0        31.0
4  2026-09-10   Tarde    M-01        1300        147.0        34.9
5  2026-09-11   Tarde    m-02        1171          NaN        30.9
6  2026-09-12   TARDE    M-03        1250        100.0        23.3
7  13/09/2026   Tarde    M-04        1027        107.0        34.6
8  2026-09-14   Noche    M-01         965         60.0        41.4
9  2026-09-15   Noche    M-02        1558        172.0        99.7

Tipos:
<class 'pandas.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   fecha        297 non-null    str    
 1 

In [2]:
def diagnostico_calidad(tabla, nombre="Dataset"):
    total_celdas = tabla.shape[0] * tabla.shape[1]
    vacios = tabla.isnull().sum().sum()
    duplicados = tabla.duplicated().sum()
    
    print(f"\n?? {nombre}:")
    print(f"   Filas: {tabla.shape[0]}")
    print(f"   Columnas: {tabla.shape[1]}")
    print(f"   Celdas vac?as: {vacios} ({vacios/total_celdas*100:.1f}%)")
    print(f"   Filas duplicadas: {duplicados}")
    print(f"   Tipos de datos:")
    for col in tabla.columns:
        print(f"      {col}: {tabla[col].dtype} ({tabla[col].notna().sum()} no vac?as)")
    
    return {"filas": tabla.shape[0], "vacios": vacios, "duplicados": duplicados}

In [3]:
stats_antes = diagnostico_calidad(df, "ANTES de limpiar")


?? ANTES de limpiar:
   Filas: 297
   Columnas: 6
   Celdas vac?as: 10 (0.6%)
   Filas duplicadas: 11
   Tipos de datos:
      fecha: str (297 no vac?as)
      turno: str (297 no vac?as)
      maquina: str (297 no vac?as)
      producidas: int64 (297 no vac?as)
      defectuosas: float64 (287 no vac?as)
      temperatura: str (297 no vac?as)


In [4]:
bitacora = []

def registrar_paso(num, accion, filas_antes, filas_despues, cambios=0):
    bitacora.append({
        "Paso": num,
        "Acci?n": accion,
        "Filas antes": filas_antes,
        "Filas despu?s": filas_despues,
        "Eliminadas": filas_antes - filas_despues,
        "Valores corregidos": cambios,
    })
    print(f"? Paso {num}: {accion}")
    print(f"   Filas: {filas_antes} ? {filas_despues} | Corregidas: {cambios}")

In [5]:
limpio = df.copy()
print(f"Copia lista: {limpio.shape}")

Copia lista: (297, 6)


In [6]:
print("PASO 1: Quitar Duplicados")
print("=" * 60)
antes = len(limpio)
print(f"Duplicados encontrados: {limpio.duplicated().sum()}")
limpio = limpio.drop_duplicates().reset_index(drop=True)
registrar_paso(1, "Quitar filas duplicadas", antes, len(limpio))

PASO 1: Quitar Duplicados
Duplicados encontrados: 11
? Paso 1: Quitar filas duplicadas
   Filas: 297 ? 286 | Corregidas: 0


In [7]:
print("\nPASO 2: Homologar M?quinas y Turnos")
print("=" * 60)
maquina_antes = limpio["maquina"].copy()
turno_antes = limpio["turno"].copy()

limpio["maquina"] = (limpio["maquina"]
                     .str.strip()
                     .str.upper()
                     .str.replace("M0", "M-0", regex=False))

limpio["turno"] = (limpio["turno"]
                   .str.strip()
                   .str.capitalize()
                   .str.replace("Manana", "Ma?ana"))

cambios_maq = (limpio["maquina"] != maquina_antes).sum()
cambios_tur = (limpio["turno"] != turno_antes).sum()

print(f"M?quinas ?nicas: {limpio['maquina'].nunique()}")
print(f"Valores: {sorted(limpio['maquina'].unique())}")
print(f"Turnos ?nicos: {limpio['turno'].nunique()}")
print(f"Valores: {sorted(limpio['turno'].unique())}")

registrar_paso(2, "Homologar m?quinas y turnos", len(limpio), len(limpio), cambios_maq + cambios_tur)


PASO 2: Homologar M?quinas y Turnos
M?quinas ?nicas: 4
Valores: ['M-01', 'M-02', 'M-03', 'M-04']
Turnos ?nicos: 3
Valores: ['Mañana', 'Noche', 'Tarde']
? Paso 2: Homologar m?quinas y turnos
   Filas: 286 ? 286 | Corregidas: 106


In [8]:
print("\nPASO 3: Convertir Tipos de Datos")
print("=" * 60)

def parse_fecha_mixta(valor):
    if pd.isna(valor):
        return pd.NaT
    texto = str(valor).strip()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            return pd.to_datetime(texto, format=fmt)
        except ValueError:
            continue
    return pd.NaT

limpio["temperatura"] = pd.to_numeric(
    limpio["temperatura"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)
limpio["fecha"] = limpio["fecha"].map(parse_fecha_mixta)

print(f"Temperatura dtype: {limpio['temperatura'].dtype}")
print(f"Fecha dtype: {limpio['fecha'].dtype}")
print(f"Vac?os temperatura: {limpio['temperatura'].isna().sum()}")
print(f"Vac?os fecha: {limpio['fecha'].isna().sum()}")

registrar_paso(3, "Convertir tipos (temperatura a n?mero, fechas)", len(limpio), len(limpio))


PASO 3: Convertir Tipos de Datos
Temperatura dtype: float64
Fecha dtype: datetime64[us]
Vac?os temperatura: 0
Vac?os fecha: 0
? Paso 3: Convertir tipos (temperatura a n?mero, fechas)
   Filas: 286 ? 286 | Corregidas: 0


In [9]:
print("\nPASO 4: Normalizar Unidades (?F ? ?C)")
print("=" * 60)

en_fahrenheit = limpio["temperatura"] > 60
print(f"Temperaturas en ?F encontradas: {en_fahrenheit.sum()}")

if en_fahrenheit.sum() > 0:
    limpio.loc[en_fahrenheit, "temperatura"] = (
        (limpio.loc[en_fahrenheit, "temperatura"] - 32) * 5 / 9
    ).round(1)
    print("Ejemplos convertidos:")
    print(limpio[en_fahrenheit][["maquina", "temperatura"]].head())

print(f"Rango final: {limpio['temperatura'].min():.1f} - {limpio['temperatura'].max():.1f} ?C")
registrar_paso(4, "Convertir ?F a ?C", len(limpio), len(limpio), int(en_fahrenheit.sum()))


PASO 4: Normalizar Unidades (?F ? ?C)
Temperaturas en ?F encontradas: 32
Ejemplos convertidos:
   maquina  temperatura
0     M-01         39.2
9     M-02         37.6
18    M-03         27.8
27    M-04         30.7
36    M-01         30.9
Rango final: 22.1 - 42.0 ?C
? Paso 4: Convertir ?F a ?C
   Filas: 286 ? 286 | Corregidas: 32


In [10]:
print("\nPASO 5A: Eliminar Faltantes")
print("=" * 60)

antes = len(limpio)
limpio = limpio.dropna(subset=["defectuosas"]).reset_index(drop=True)
registrar_paso(5, "Eliminar filas sin defectuosas", antes, len(limpio))


PASO 5A: Eliminar Faltantes
? Paso 5: Eliminar filas sin defectuosas
   Filas: 286 ? 276 | Corregidas: 0


In [11]:
print("\nPASO 5B: Imputar Temperatura con Mediana por M?quina")
print("=" * 60)

if "temp_imputada" not in limpio.columns:
    limpio["temp_imputada"] = limpio["temperatura"].isna()

print(f"Temperaturas a imputar: {limpio['temp_imputada'].sum()}")

mediana_por_maq = limpio.groupby("maquina")["temperatura"].transform("median")
limpio["temperatura"] = limpio["temperatura"].fillna(mediana_por_maq)

print(f"Vac?os en tabla despu?s: {limpio.isnull().sum().sum()}")
registrar_paso(5, "Imputar temperatura por m?quina", len(limpio), len(limpio), int(limpio["temp_imputada"].sum()))


PASO 5B: Imputar Temperatura con Mediana por M?quina
Temperaturas a imputar: 0
Vac?os en tabla despu?s: 0
? Paso 5: Imputar temperatura por m?quina
   Filas: 276 ? 276 | Corregidas: 0


In [12]:
print("\nPASO 6: Eliminar Valores Imposibles")
print("=" * 60)

sobre_cap = limpio["producidas"] > 1800
invertidas = limpio["defectuosas"] > limpio["producidas"]
negativas = limpio["defectuosas"] < 0

print(f"Producidas > 1800: {sobre_cap.sum()}")
print(f"Defectuosas > producidas: {invertidas.sum()}")
print(f"Defectuosas negativas: {negativas.sum()}")

imposible = sobre_cap | invertidas | negativas
print(f"Total imposibles: {imposible.sum()}")

antes = len(limpio)
limpio = limpio[~imposible].reset_index(drop=True)
limpio["producidas"] = limpio["producidas"].astype(int)
limpio["defectuosas"] = limpio["defectuosas"].astype(int)

registrar_paso(6, "Eliminar valores imposibles", antes, len(limpio))


PASO 6: Eliminar Valores Imposibles
Producidas > 1800: 4
Defectuosas > producidas: 5
Defectuosas negativas: 0
Total imposibles: 5
? Paso 6: Eliminar valores imposibles
   Filas: 276 ? 271 | Corregidas: 0


In [13]:
print("\n" + "=" * 60)
print("BIT?CORA COMPLETA DE LIMPIEZA")
print("=" * 60)
bitacora_df = pd.DataFrame(bitacora)
print(bitacora_df.to_string(index=False))


BIT?CORA COMPLETA DE LIMPIEZA
 Paso                                         Acci?n  Filas antes  Filas despu?s  Eliminadas  Valores corregidos
    1                        Quitar filas duplicadas          297            286          11                   0
    2                    Homologar m?quinas y turnos          286            286           0                 106
    3 Convertir tipos (temperatura a n?mero, fechas)          286            286           0                   0
    4                              Convertir ?F a ?C          286            286           0                  32
    5                 Eliminar filas sin defectuosas          286            276          10                   0
    5                Imputar temperatura por m?quina          276            276           0                   0
    6                    Eliminar valores imposibles          276            271           5                   0


In [14]:
stats_despues = diagnostico_calidad(limpio, "DESPU?S de limpiar")


?? DESPU?S de limpiar:
   Filas: 271
   Columnas: 7
   Celdas vac?as: 0 (0.0%)
   Filas duplicadas: 0
   Tipos de datos:
      fecha: datetime64[us] (271 no vac?as)
      turno: str (271 no vac?as)
      maquina: str (271 no vac?as)
      producidas: int64 (271 no vac?as)
      defectuosas: int64 (271 no vac?as)
      temperatura: float64 (271 no vac?as)
      temp_imputada: bool (271 no vac?as)


In [15]:
print("\n" + "=" * 60)
print("COMPARACI?N ANTES/DESPU?S")
print("=" * 60)
comparacion = pd.DataFrame({
    "M?trica": ["Filas", "Celdas vac?as", "Duplicados"],
    "Antes": [stats_antes["filas"], stats_antes["vacios"], stats_antes["duplicados"]],
    "Despu?s": [stats_despues["filas"], stats_despues["vacios"], stats_despues["duplicados"]],
})
print(comparacion.to_string(index=False))


COMPARACI?N ANTES/DESPU?S
      M?trica  Antes  Despu?s
        Filas    297      271
Celdas vac?as     10        0
   Duplicados     11        0


In [16]:
print("\n" + "=" * 60)
print("CONSULTA 1: Tasa de Defectos por M?quina")
print("=" * 60)

limpio["tasa_defectos"] = (limpio["defectuosas"] / limpio["producidas"] * 100).round(2)

por_maquina = limpio.groupby("maquina").agg({
    "producidas": "sum",
    "defectuosas": "sum",
    "temperatura": "mean",
}).round(2)

por_maquina["tasa_%"] = (por_maquina["defectuosas"] / por_maquina["producidas"] * 100).round(2)
por_maquina = por_maquina.sort_values("tasa_%", ascending=False)

print(por_maquina)

colores = ["#c0392b" if i == 0 else "#95a5a6" for i in range(len(por_maquina))]
por_maquina["tasa_%"].plot(kind="bar", color=colores, figsize=(8, 4))
plt.title("Tasa de Defectos por M?quina")
plt.ylabel("% de defectuosas")
plt.xlabel("M?quina")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

peor_maq = por_maquina["tasa_%"].idxmax()
peor_tasa = por_maquina["tasa_%"].max()
print(f"\n?? HALLAZGO 1:")
print(f"   La m?quina {peor_maq} tiene la mayor tasa de defectos: {peor_tasa}%")
print(f"   Recomendaci?n: Enviar {peor_maq} a mantenimiento como prioridad.")


CONSULTA 1: Tasa de Defectos por M?quina
         producidas  defectuosas  temperatura  tasa_%
maquina                                              
M-01          88155         7020        31.84    7.96
M-03          89870         6924        32.54    7.70
M-02          92384         7033        31.11    7.61
M-04          86765         5928        31.88    6.83



?? HALLAZGO 1:
   La m?quina M-01 tiene la mayor tasa de defectos: 7.96%
   Recomendaci?n: Enviar M-01 a mantenimiento como prioridad.


C:\Users\Manuel\AppData\Local\Temp\ipykernel_40728\1210233475.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
print("\n" + "=" * 60)
print("CONSULTA 2: Correlaci?n Temperatura vs Defectos")
print("=" * 60)

por_maquina_corr = limpio.groupby("maquina").apply(
    lambda x: x["temperatura"].corr(x["tasa_defectos"])
).round(3)

print("Correlaci?n (temperatura vs tasa defectos):")
print(por_maquina_corr)

fig, ax = plt.subplots(figsize=(10, 5))

for maq in limpio["maquina"].unique():
    datos = limpio[limpio["maquina"] == maq]
    ax.scatter(datos["temperatura"], datos["tasa_defectos"], label=maq, alpha=0.6, s=100)

plt.title("Temperatura vs Tasa de Defectos")
plt.xlabel("Temperatura (?C)")
plt.ylabel("% Defectuosas")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

max_corr_maq = por_maquina_corr.abs().idxmax()
max_corr_val = por_maquina_corr[max_corr_maq]
print(f"\n?? HALLAZGO 2:")
print(f"   M?quina {max_corr_maq} muestra correlaci?n: {max_corr_val}")
if abs(max_corr_val) > 0.5:
    print(f"   ? Relaci?n FUERTE: temperatura influye en defectos")
else:
    print(f"   ? Relaci?n D?BIL: otros factores son m?s importantes")


CONSULTA 2: Correlaci?n Temperatura vs Defectos
Correlaci?n (temperatura vs tasa defectos):
maquina
M-01   -0.006
M-02   -0.124
M-03    0.079
M-04   -0.121
dtype: float64

?? HALLAZGO 2:
   M?quina M-02 muestra correlaci?n: -0.124
   ? Relaci?n D?BIL: otros factores son m?s importantes


C:\Users\Manuel\AppData\Local\Temp\ipykernel_40728\1081775962.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
print("\n" + "=" * 60)
print("EXPORTAR DATOS")
print("=" * 60)

limpio.to_csv("registro_produccion_limpio.csv", index=False)
print("? Archivo guardado: registro_produccion_limpio.csv")
print(f"   Filas finales: {len(limpio)}")
print(f"   Columnas: {limpio.columns.tolist()}")

print("\n? TRABAJO COMPLETADO EXITOSAMENTE ?")


EXPORTAR DATOS
? Archivo guardado: registro_produccion_limpio.csv
   Filas finales: 271
   Columnas: ['fecha', 'turno', 'maquina', 'producidas', 'defectuosas', 'temperatura', 'temp_imputada', 'tasa_defectos']

? TRABAJO COMPLETADO EXITOSAMENTE ?
